# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a walk-through for loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Dataset identifier: {metadata.identifier}")
print(f"Published on: {metadata.datePublished}")
print(f"Available keywords: {metadata.keywords}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

**All entities are referenced strictly by their `@id` fields.**

In [ ]:
# List available record sets.
record_sets = list(dataset.record_sets)
print("Available Record Sets (by @id):")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', 'No name provided')}")

# For each record set, list its fields by their @id
for rs in record_sets:
    print(f"\nFields for Record Set {rs['@id']}:")
    for field in rs['fields']:
        print(f"  - Field @id: {field['@id']} | Name: {field.get('name', '')} | DataType: {field.get('dataType', '')}")

## 3. Data Extraction
Load records from all available record sets into DataFrames. Use each record set and field `@id`.

In [ ]:
# Organize record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

# Load each record set into a pandas DataFrame
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns for Record Set {record_set_id}:")
    print(df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Explore and preprocess tabular data: filter records, normalize numeric values, and group by key fields.

**Note:** All fields/columns are referenced strictly by their `@id`. Replace numeric and grouping fields below with actual `@id`s as identified in section 2.

In [ ]:
# Example using Table 1 (demographics and clinical features)

# Select which record set to explore
table1_record_set_id = record_set_ids[0]
df1 = dataframes[table1_record_set_id]

# Find a numeric field for demonstration (e.g., age at diagnosis)
# From data overview, suppose age_at_diagnosis_field_id is the @id for age
age_at_diagnosis_field_id = None
for rs in record_sets:
    if rs['@id'] == table1_record_set_id:
        for field in rs['fields']:
            if 'age' in field.get('name', '').lower():
                age_at_diagnosis_field_id = field['@id']
                break
        break

if age_at_diagnosis_field_id is None:
    print("No age field found in Table 1.")
else:
    # Filter for patients with age at diagnosis > 25
    threshold = 25
    filtered_df = df1[df1[age_at_diagnosis_field_id] > threshold]
    print(f"Filtered records with {age_at_diagnosis_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize age
    filtered_df[f"{age_at_diagnosis_field_id}_normalized"] = (
        filtered_df[age_at_diagnosis_field_id] - filtered_df[age_at_diagnosis_field_id].mean()
    ) / filtered_df[age_at_diagnosis_field_id].std()
    print(f"Normalized {age_at_diagnosis_field_id} for filtered records:")
    print(filtered_df[[age_at_diagnosis_field_id, f"{age_at_diagnosis_field_id}_normalized" ]].head())

    # Group by a categorical clinical feature (e.g., hirsutism)
    hirsutism_field_id = None
    for field in rs['fields']:
        if 'hirsutism' in field.get('name', '').lower():
            hirsutism_field_id = field['@id']
            break

    if hirsutism_field_id and hirsutism_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(hirsutism_field_id)[age_at_diagnosis_field_id].mean()
        print(f"Mean age at diagnosis grouped by {hirsutism_field_id}:")
        print(grouped_df)

## 5. Visualization
Visualize distributions and relationships between clinical or genetic features using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of age at diagnosis
if age_at_diagnosis_field_id:
    plt.figure(figsize=(6, 4))
    sns.histplot(df1[age_at_diagnosis_field_id], bins=5, kde=True)
    plt.title('Distribution of Age at Diagnosis')
    plt.xlabel(age_at_diagnosis_field_id)
    plt.ylabel('Frequency')
    plt.show()

# Scatter: Age at diagnosis vs Height (if available)
height_field_id = None
for field in rs['fields']:
    if 'height' in field.get('name', '').lower():
        height_field_id = field['@id']
        break

if age_at_diagnosis_field_id and height_field_id:
    plt.figure(figsize=(6, 4))
    sns.scatterplot(
        x=df1[age_at_diagnosis_field_id],
        y=df1[height_field_id],
        hue=df1[hirsutism_field_id] if hirsutism_field_id else None
    )
    plt.title('Age at Diagnosis vs. Height')
    plt.xlabel(age_at_diagnosis_field_id)
    plt.ylabel(height_field_id)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from this FAIR^2 dataset exploration.

- The FAIR^2 dataset includes clinical, hormonal, and genetic information for three female patients diagnosed with non-classical 21-hydroxylase deficiency-associated infertility.
- Data is organized in multiple record sets (e.g., Table 1: demographics/clinical features, Table 2: hormone/imaging, Table 3: genetic variants), all referenced via their `@id`s following Croissant schema.
- Exploratory analysis illustrates heterogeneity in age at diagnosis and other clinical traits. Grouping and normalization demonstrate potential for custom cohort analysis, but small sample size highlights limitations for statistical inference.
- Visualizations further showcase distributions and clinical correlations within the dataset.
- **Note:** All data processing strictly adhered to referencing fields, columns, and record sets by their `@id`, facilitating robust and reproducible data handling in the Croissant ecosystem.